# Unified Bharat: Medallion Pipeline

Bronze → Silver → Gold Lakehouse Architecture

**Datasets:**
- CSR Spending (Ministry of Corporate Affairs)
- Groundwater Quality (Ministry of Jal Shakti)
- Institutions (Education/Technical)
- LGD Master District Codes
- State Population (2024 estimates)

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, upper, regexp_extract, trim, lower, sum as spark_sum,
    avg, count, first, lit, when, coalesce, lag, expr,
    count as spark_count, round as spark_round
)
from pyspark.sql.window import Window
import pandas as pd
import json
from datetime import datetime

# Track data lineage
lineage = []

def log_step(layer, table, rows_in, rows_out, transform, confidence):
    pct = round((rows_out/rows_in)*100, 2) if rows_in > 0 else 0
    lineage.append({
        'layer': layer,
        'table': table,
        'rows_in': rows_in,
        'rows_out': rows_out,
        'pct_retained': pct,
        'transformation': transform,
        'confidence': confidence
    })
    print(f"[{layer}] {table}: {rows_in} → {rows_out} rows ({pct}% retained) | {confidence}")

packages = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4"
]

spark = SparkSession.builder \
    .appName("UnifiedBharat_Pipeline") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://unified-bharat/warehouse") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "supersecretpassword") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Running. Version:", spark.version)

Spark Running. Version: 3.5.0


## Bronze Layer: Raw Ingestion

In [2]:
# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

# Load raw CSVs
bronze_csr = spark.read.csv("/home/jovyan/data/raw/csr_district.csv", header=True, inferSchema=True)
bronze_water = spark.read.csv("/home/jovyan/data/raw/ground_water.csv", header=True, inferSchema=True)
bronze_inst = spark.read.csv("/home/jovyan/data/raw/institutions.csv", header=True, inferSchema=True)
bronze_lgd = spark.read.csv("/home/jovyan/data/raw/LGD.csv", header=True, inferSchema=True)
bronze_pop = spark.read.csv("/home/jovyan/data/raw/population.csv", header=True, inferSchema=True)

# Register bronze tables
bronze_csr.write.format("iceberg").mode("overwrite").saveAsTable("local.bronze.csr_spending")
bronze_water.write.format("iceberg").mode("overwrite").saveAsTable("local.bronze.ground_water")
bronze_inst.write.format("iceberg").mode("overwrite").saveAsTable("local.bronze.institutions")
bronze_lgd.write.format("iceberg").mode("overwrite").saveAsTable("local.bronze.lgd_master")
bronze_pop.write.format("iceberg").mode("overwrite").saveAsTable("local.bronze.population")

print("Bronze tables created successfully!")
print(f"CSR: {bronze_csr.count()} rows")
print(f"Groundwater: {bronze_water.count()} rows")
print(f"Institutions: {bronze_inst.count()} rows")
print(f"LGD: {bronze_lgd.count()} rows")
print(f"Population: {bronze_pop.count()} rows")

Bronze tables created successfully!
CSR: 28833 rows
Groundwater: 188208 rows
Institutions: 2140 rows
LGD: 784 rows
Population: 36 rows


## Silver Layer: CSR Processing

**Transformation:** District-year → State-year
- Extract year from text
- Standardize names
- Convert INR to USD
- **AVERAGE** across districts/departments (not sum)

In [3]:
csr_raw = spark.read.format("iceberg").load("local.bronze.csr_spending")
csr_in = csr_raw.count()

silver_csr = (
    csr_raw
    .withColumn("year", regexp_extract(col("Year"), r"(\d{4})", 1).cast("int"))
    .withColumn("state_name", upper(trim(col("StateName"))))
    .withColumn("csir_spent_inr_crores", col("`I6008_6.sum`").cast("double"))
    .withColumn("csir_spent_usd_millions", (col("`I6008_6.sum`") * 10 / 83.0).cast("double"))
    .filter(col("year").isNotNull())
    .groupBy("state_name", "year")
    .agg(
        avg("csir_spent_inr_crores").alias("avg_csr_inr_crores"),
        avg("csir_spent_usd_millions").alias("avg_csr_usd_millions"),
        count("*").alias("district_records_aggregated")
    )
    .orderBy("state_name", "year")
)

silver_csr.write.format("iceberg").mode("overwrite").saveAsTable("local.silver.csr_state_year")
csr_out = silver_csr.count()
log_step("Silver", "csr_state_year", csr_in, csr_out, "District→State AVERAGE, year extract, currency convert", "High")
silver_csr.show(5)

[Silver] csr_state_year: 28833 → 245 rows (0.85% retained) | High
+--------------------+----+--------------------+--------------------+---------------------------+
|          state_name|year|  avg_csr_inr_crores|avg_csr_usd_millions|district_records_aggregated|
+--------------------+----+--------------------+--------------------+---------------------------+
|ANDAMAN AND NICOB...|2014| 0.03222222222222222|0.003882195448460508|                          9|
|ANDAMAN AND NICOB...|2015|0.061111111111111116|0.007362784471218207|                          9|
|ANDAMAN AND NICOB...|2016|                0.07|0.008433734939759038|                          9|
|ANDAMAN AND NICOB...|2017|  0.0811111111111111|0.009772423025435075|                          9|
|ANDAMAN AND NICOB...|2018| 0.09000000000000001|0.010843373493975905|                          9|
+--------------------+----+--------------------+--------------------+---------------------------+
only showing top 5 rows



## Silver Layer: Groundwater Processing

**Transformation:** Station-level → State-year
- Standardize state names
- Match districts to LGD master
- **Keep only high-coverage parameters (>65%):** Hardness, PH, Nitrate, Fluoride
- **DROP low-coverage:** TDS (21%), Iron (34%), Arsenic (9%)
- Compute contamination_index (0-4 scale)

In [4]:
water_raw = spark.read.format("iceberg").load("local.bronze.ground_water")
water_in = water_raw.count()
lgd_raw = spark.read.format("iceberg").load("local.bronze.lgd_master")

# Standardize state names
water_std = (
    water_raw
    .withColumn("state_name", upper(trim(col("srcStateName"))))
    .withColumn("district_name_raw", trim(col("srcDistrictName")))
    .withColumn("year", col("srcYear").cast("int"))
)

# Prepare LGD reference
lgd_ref = (
    lgd_raw
    .select(
        upper(trim(col("District Name (In English)"))).alias("lgd_district"),
        col("District LGD Code").cast("string").alias("lgd_district_code")
    )
    .dropDuplicates(["lgd_district"])
)

# Simple exact match first (case-insensitive)
water_matched = (
    water_std
    .join(lgd_ref, lower(water_std.district_name_raw) == lower(lgd_ref.lgd_district), "left")
)

# Aggregate to state-year level - KEEP ONLY HIGH-COVERAGE PARAMS
silver_water = (
    water_matched
    .groupBy("state_name", "year")
    .agg(
        avg("Amount of Hardness Total").alias("hardness_avg"),
        avg("Amount of Potential of Hydrogen").alias("ph_avg"),
        avg("Amount of Nitrate").alias("nitrate_avg"),
        avg("Amount of Fluorine").alias("fluoride_avg"),
        count("*").alias("num_stations"),
        spark_sum(when(col("lgd_district_code").isNotNull(), 1).otherwise(0)).alias("matched_districts"),
        count("*").alias("total_district_records")
    )
    .withColumn("contamination_index",
        when(col("hardness_avg").isNotNull() & (col("hardness_avg") > 200), 1).otherwise(0) +
        when(col("ph_avg").isNotNull() & ((col("ph_avg") < 6.5) | (col("ph_avg") > 8.5)), 1).otherwise(0) +
        when(col("nitrate_avg").isNotNull() & (col("nitrate_avg") > 45), 1).otherwise(0) +
        when(col("fluoride_avg").isNotNull() & (col("fluoride_avg") > 1.5), 1).otherwise(0)
    )
    .withColumn("match_confidence_pct",
        spark_round((col("matched_districts") / col("total_district_records")) * 100, 2)
    )
    .select("state_name", "year", "hardness_avg", "ph_avg", "nitrate_avg", "fluoride_avg",
            "contamination_index", "match_confidence_pct")
    .orderBy("state_name", "year")
)

silver_water.write.format("iceberg").mode("overwrite").saveAsTable("local.silver.groundwater_state_year")
water_out = silver_water.count()
log_step("Silver", "groundwater_state_year", water_in, water_out,
         "Station→State-Year, DROP TDS/Iron/Arsenic (<65%), KEEP Hardness/PH/Nitrate/Fluoride, LGD match",
         "High (only >65% coverage params kept)")
silver_water.show(5)

[Silver] groundwater_state_year: 188208 → 248 rows (0.13% retained) | High (only >65% coverage params kept)
+-----------------+----+------------------+-----------------+-----------------+-------------------+-------------------+--------------------+
|       state_name|year|      hardness_avg|           ph_avg|      nitrate_avg|       fluoride_avg|contamination_index|match_confidence_pct|
+-----------------+----+------------------+-----------------+-----------------+-------------------+-------------------+--------------------+
|ANDAMAN & NICOBAR|2011|207.57142857142858|7.774285714285711|2.466666666666667|0.18599999999999997|                  1|                 0.0|
|ANDAMAN & NICOBAR|2012| 197.7027027027027| 8.21081081081081|            7.625|            0.14335|                  0|                 0.0|
|ANDAMAN & NICOBAR|2013| 189.4262295081967|7.937704918032784|6.133454545454545| 0.3391999999999999|                  0|                 0.0|
|ANDAMAN & NICOBAR|2014|              NULL|   

## Silver Layer: Institution Processing

**Transformation:** State-institution-type-year → State-year
- Standardize names, extract year
- Fill nulls: numeric=0, categorical='Unknown'
- **AVERAGE** across institution types (not sum)

In [5]:
inst_raw = spark.read.format("iceberg").load("local.bronze.institutions")
inst_in = inst_raw.count()

# Check for nulls before cleaning
null_counts = inst_raw.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in inst_raw.columns])
print("Null counts before cleaning:")
null_counts.show()

silver_inst = (
    inst_raw
    .withColumn("state_name", upper(trim(col("State"))))
    .withColumn("year", regexp_extract(col("Year"), r"(\d{4})", 1).cast("int"))
    .withColumn("type_of_institution", coalesce(trim(col("Type Of Institution")), lit("Unknown")))
    .withColumn("approved_intake", coalesce(col("Approved Intake (UOM:Number), Scaling Factor:1").cast("double"), lit(0.0)))
    .withColumn("institutions", coalesce(col("Institutions (UOM:Number), Scaling Factor:1").cast("double"), lit(0.0)))
    .withColumn("total_approved_institutions", coalesce(col("Total Approved Institutions (UOM:Number), Scaling Factor:1").cast("double"), lit(0.0)))
    .filter(col("year").isNotNull())
    .groupBy("state_name", "year")
    .agg(
        avg("approved_intake").alias("avg_approved_intake"),
        avg("institutions").alias("avg_institutions"),
        avg("total_approved_institutions").alias("avg_total_approved_institutions"),
        count("*").alias("institution_types_collapsed")
    )
    .orderBy("state_name", "year")
)

silver_inst.write.format("iceberg").mode("overwrite").saveAsTable("local.silver.institutions_state_year")
inst_out = silver_inst.count()
log_step("Silver", "institutions_state_year", inst_in, inst_out,
         "Type collapse AVERAGE, null fill(0), year extract", "High")
silver_inst.show(5)

Null counts before cleaning:
+-------+-----+----+------+-------+-------------------+----------------------------------------------+-------------------------------------------+----------------------------------------------------------+
|Country|State|Year|Region|Program|Type Of Institution|Approved Intake (UOM:Number), Scaling Factor:1|Institutions (UOM:Number), Scaling Factor:1|Total Approved Institutions (UOM:Number), Scaling Factor:1|
+-------+-----+----+------+-------+-------------------+----------------------------------------------+-------------------------------------------+----------------------------------------------------------+
|      0|    0|   0|     0|    632|                  0|                                             3|                                          4|                                                      1122|
+-------+-----+----+------+-------+-------------------+----------------------------------------------+-------------------------------------------+-

## Silver Layer: Population Processing

In [6]:
pop_raw = spark.read.format("iceberg").load("local.bronze.population")
pop_in = pop_raw.count()

silver_pop = (
    pop_raw
    .withColumn("state_name", upper(trim(col("state_name"))))
    .select("state_name", "population_2024_estimate")
)

silver_pop.write.format("iceberg").mode("overwrite").saveAsTable("local.silver.population_state")
pop_out = silver_pop.count()
log_step("Silver", "population_state", pop_in, pop_out, "Name standardization", "High")
silver_pop.show(10)

[Silver] population_state: 36 → 36 rows (100.0% retained) | High
+--------------+------------------------+
|    state_name|population_2024_estimate|
+--------------+------------------------+
| UTTAR PRADESH|               241066874|
|         BIHAR|               128592000|
|   MAHARASHTRA|               127528000|
|   WEST BENGAL|                99563000|
|MADHYA PRADESH|                87610000|
|     RAJASTHAN|                81897000|
|    TAMIL NADU|                77089000|
|     KARNATAKA|                68115000|
|       GUJARAT|                72367000|
|ANDHRA PRADESH|                53340000|
+--------------+------------------------+
only showing top 10 rows



## Gold Layer: Water Quality Final Clean

**Transformation:** Silver → Gold
- Drop match_confidence_pct (lineage tracked separately)
- Fill remaining nulls in chemical params with column averages
- **Recalculate contamination_index** after fill

In [7]:
water_silver = spark.read.format("iceberg").load("local.silver.groundwater_state_year")
water_silver_in = water_silver.count()

# Calculate column averages for filling nulls
avg_vals = water_silver.agg(
    avg("hardness_avg").alias("avg_hardness"),
    avg("ph_avg").alias("avg_ph"),
    avg("nitrate_avg").alias("avg_nitrate"),
    avg("fluoride_avg").alias("avg_fluoride")
).collect()[0]

avg_hardness = avg_vals["avg_hardness"]
avg_ph = avg_vals["avg_ph"]
avg_nitrate = avg_vals["avg_nitrate"]
avg_fluoride = avg_vals["avg_fluoride"]

print(f"Column averages for null fill:")
print(f"  Hardness: {avg_hardness:.2f}")
print(f"  PH: {avg_ph:.2f}")
print(f"  Nitrate: {avg_nitrate:.2f}")
print(f"  Fluoride: {avg_fluoride:.2f}")

# Count nulls before fill
nulls_before = water_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in ["hardness_avg", "ph_avg", "nitrate_avg", "fluoride_avg"]
]).collect()[0]
total_nulls = sum(nulls_before.asDict().values())
print(f"\nNull values before fill: {total_nulls}")

gold_water = (
    water_silver
    .withColumn("hardness_avg", coalesce(col("hardness_avg"), lit(avg_hardness)))
    .withColumn("ph_avg", coalesce(col("ph_avg"), lit(avg_ph)))
    .withColumn("nitrate_avg", coalesce(col("nitrate_avg"), lit(avg_nitrate)))
    .withColumn("fluoride_avg", coalesce(col("fluoride_avg"), lit(avg_fluoride)))
    .drop("match_confidence_pct")
    .withColumn("contamination_index",
        when(col("hardness_avg") > 200, 1).otherwise(0) +
        when((col("ph_avg") < 6.5) | (col("ph_avg") > 8.5), 1).otherwise(0) +
        when(col("nitrate_avg") > 45, 1).otherwise(0) +
        when(col("fluoride_avg") > 1.5, 1).otherwise(0)
    )
    .orderBy("state_name", "year")
)

gold_water.write.format("iceberg").mode("overwrite").saveAsTable("local.gold.groundwater_state_year")
water_gold_out = gold_water.count()
log_step("Gold", "groundwater_state_year", water_silver_in, water_gold_out,
         f"Drop match_confidence, fill {total_nulls} nulls with column averages, recalc contamination_index (0-4)",
         "Very High (all nulls imputed, only high-coverage params)")
gold_water.show(5)

Column averages for null fill:
  Hardness: 265.30
  PH: 8.02
  Nitrate: 29.44
  Fluoride: 0.66

Null values before fill: 108
[Gold] groundwater_state_year: 248 → 248 rows (100.0% retained) | Very High (all nulls imputed, only high-coverage params)
+-----------------+----+------------------+-----------------+------------------+-------------------+-------------------+
|       state_name|year|      hardness_avg|           ph_avg|       nitrate_avg|       fluoride_avg|contamination_index|
+-----------------+----+------------------+-----------------+------------------+-------------------+-------------------+
|ANDAMAN & NICOBAR|2011|207.57142857142858|7.774285714285711| 2.466666666666667|0.18599999999999997|                  1|
|ANDAMAN & NICOBAR|2012| 197.7027027027027| 8.21081081081081|             7.625|            0.14335|                  0|
|ANDAMAN & NICOBAR|2013| 189.4262295081967|7.937704918032784| 6.133454545454545| 0.3391999999999999|                  0|
|ANDAMAN & NICOBAR|2014|26

## Gold Layer: Unified State-Year Panel

In [8]:
csr_s = spark.read.format("iceberg").load("local.silver.csr_state_year")
water_g = spark.read.format("iceberg").load("local.gold.groundwater_state_year")
inst_s = spark.read.format("iceberg").load("local.silver.institutions_state_year")
pop_s = spark.read.format("iceberg").load("local.silver.population_state")

# Join all tables
panel = (
    csr_s
    .join(water_g, ["state_name", "year"], "outer")
    .join(inst_s, ["state_name", "year"], "outer")
    .join(pop_s, ["state_name"], "left")
)

# Create lag variable and derived metrics
window_spec = Window.partitionBy("state_name").orderBy("year")

gold_panel = (
    panel
    .withColumn("avg_csr_inr_crores", coalesce(col("avg_csr_inr_crores"), lit(0.0)))
    .withColumn("csr_spent_lag1", lag("avg_csr_inr_crores", 1).over(window_spec))
    .withColumn("csr_per_capita_inr",
        when(col("population_2024_estimate").isNotNull() & (col("population_2024_estimate") > 0),
             (col("avg_csr_inr_crores") * 10000000) / col("population_2024_estimate"))
        .otherwise(lit(None)))
    .withColumn("institutions_per_million",
        when(col("population_2024_estimate").isNotNull() & (col("population_2024_estimate") > 0),
             (col("avg_institutions") * 1000000) / col("population_2024_estimate"))
        .otherwise(lit(None)))
    .withColumn("has_csr", when(col("avg_csr_inr_crores").isNotNull() & (col("avg_csr_inr_crores") > 0), 1).otherwise(0))
    .withColumn("has_groundwater", when(col("contamination_index").isNotNull(), 1).otherwise(0))
    .withColumn("has_institutions", when(col("avg_institutions").isNotNull() & (col("avg_institutions") > 0), 1).otherwise(0))
    .withColumn("panel_completeness", col("has_csr") + col("has_groundwater") + col("has_institutions"))
    .select(
        "state_name", "year",
        "avg_csr_inr_crores", "avg_csr_usd_millions", "csr_spent_lag1", "csr_per_capita_inr",
        "hardness_avg", "ph_avg", "nitrate_avg", "fluoride_avg", "contamination_index",
        "avg_approved_intake", "avg_institutions", "avg_total_approved_institutions", "institutions_per_million",
        "population_2024_estimate", "panel_completeness"
    )
    .orderBy("state_name", "year")
)

gold_panel.write.format("iceberg").mode("overwrite").saveAsTable("local.gold.state_year_panel")
panel_out = gold_panel.count()

# Calculate completeness stats
complete_3 = gold_panel.filter(col("panel_completeness") == 3).count()
complete_2 = gold_panel.filter(col("panel_completeness") == 2).count()
complete_1 = gold_panel.filter(col("panel_completeness") == 1).count()
complete_0 = gold_panel.filter(col("panel_completeness") == 0).count()

print(f"\nGold Panel: {panel_out} total state-year combinations")
print(f"  Complete (all 3 datasets): {complete_3} ({round(complete_3/panel_out*100,1)}%)")
print(f"  Partial (2 datasets): {complete_2} ({round(complete_2/panel_out*100,1)}%)")
print(f"  Sparse (1 dataset): {complete_1} ({round(complete_1/panel_out*100,1)}%)")
print(f"  Empty (0 datasets): {complete_0} ({round(complete_0/panel_out*100,1)}%)")

log_step("Gold", "state_year_panel", csr_out + water_gold_out + inst_out, panel_out,
         "4-way outer join + lag + per-capita, minimal clean columns",
         "Panel completeness tracked")
gold_panel.show(10)


Gold Panel: 429 total state-year combinations
  Complete (all 3 datasets): 125 (29.1%)
  Partial (2 datasets): 162 (37.8%)
  Sparse (1 dataset): 139 (32.4%)
  Empty (0 datasets): 3 (0.7%)
[Gold] state_year_panel: 841 → 429 rows (51.01% retained) | Panel completeness tracked
+--------------------+----+-------------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-------------------+----------------+-------------------------------+------------------------+------------------------+------------------+
|          state_name|year| avg_csr_inr_crores|avg_csr_usd_millions|csr_spent_lag1|csr_per_capita_inr|      hardness_avg|           ph_avg|       nitrate_avg|       fluoride_avg|contamination_index|avg_approved_intake|avg_institutions|avg_total_approved_institutions|institutions_per_million|population_2024_estimate|panel_completeness|
+--------------------+----+-------------------+---

## Data Lineage Summary & Auto-Update

In [9]:
# Display lineage table
lineage_df = spark.createDataFrame(lineage)
lineage_df.show(truncate=False)

# Save lineage to CSV
lineage_pd = pd.DataFrame(lineage)
lineage_pd.to_csv("/home/jovyan/data/lineage_summary.csv", index=False)

# Auto-update DATA_LINEAGE.md
lineage_md = f"""# Unified Bharat: Data Lineage & Quality Report

**Auto-generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Row Count Lineage

| Layer | Table | Rows In | Rows Out | % Retained | Transformation | Confidence |
|-------|-------|---------|----------|------------|----------------|------------|
"""

for row in lineage:
    lineage_md += f"| {row['layer']} | {row['table']} | {row['rows_in']} | {row['rows_out']} | {row['pct_retained']}% | {row['transformation']} | {row['confidence']} |\n"

lineage_md += f"""

## Key Data Quality Decisions

### Groundwater (Water Quality)
- **Dropped columns** (data <65%): TDS (21%), Iron (34%), Arsenic (9%)
- **Kept columns** (data >65%): Hardness (69%), PH (75%), Nitrate (65%), Fluoride (71%)
- **Null handling**: {total_nulls} null values filled with column averages
- **Contamination index**: Recalculated 0-4 scale after fill

### CSR Spending
- **Aggregation**: AVERAGE (not sum) across districts/departments
- **Currency**: INR crores → USD millions (rate: 83.0)

### Institutions
- **Aggregation**: AVERAGE (not sum) across institution types
- **Null handling**: Numeric nulls filled with 0 before averaging

### Gold Panel Completeness
- Complete (3/3 datasets): {complete_3} rows ({round(complete_3/panel_out*100,1)}%)
- Partial (2/3 datasets): {complete_2} rows ({round(complete_2/panel_out*100,1)}%)
- Sparse (1/3 datasets): {complete_1} rows ({round(complete_1/panel_out*100,1)}%)
- Empty (0/3 datasets): {complete_0} rows ({round(complete_0/panel_out*100,1)}%)
"""

with open("/home/jovyan/work/DATA_LINEAGE.md", "w") as f:
    f.write(lineage_md)

print("\nLineage saved to:")
print("  - data/lineage_summary.csv")
print("  - work/DATA_LINEAGE.md (auto-updated)")

+--------------------------------------------------------+------+------------+-------+--------+-----------------------+----------------------------------------------------------------------------------------------+
|confidence                                              |layer |pct_retained|rows_in|rows_out|table                  |transformation                                                                                |
+--------------------------------------------------------+------+------------+-------+--------+-----------------------+----------------------------------------------------------------------------------------------+
|High                                                    |Silver|0.85        |28833  |245     |csr_state_year         |District→State AVERAGE, year extract, currency convert                                        |
|High (only >65% coverage params kept)                   |Silver|0.13        |188208 |248     |groundwater_state_year |Station→State-Year, D

## Export Gold Panel for Analysis

In [10]:
# Export to CSV for external analysis
gold_pd = gold_panel.toPandas()
gold_pd.to_csv("/home/jovyan/data/gold/gold_state_year_panel.csv", index=False)
print(f"Gold panel exported: {len(gold_pd)} rows, {len(gold_pd.columns)} columns")
print("\nColumns:")
for c in gold_pd.columns:
    print(f"  - {c}")

Gold panel exported: 429 rows, 17 columns

Columns:
  - state_name
  - year
  - avg_csr_inr_crores
  - avg_csr_usd_millions
  - csr_spent_lag1
  - csr_per_capita_inr
  - hardness_avg
  - ph_avg
  - nitrate_avg
  - fluoride_avg
  - contamination_index
  - avg_approved_intake
  - avg_institutions
  - avg_total_approved_institutions
  - institutions_per_million
  - population_2024_estimate
  - panel_completeness
